# StyleFinder: A Two-Layer Fashion Information Retrieval System

**Layer 1:** Expert style recommendations based on body type and occasion  
**Layer 2:** Real product matching from a catalog of 44,000+ fashion items

In [ ]:
%pip install --upgrade -q rank_bm25 sentence-transformers umap-learn datasets plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

from collections import defaultdict, Counter, namedtuple
import math

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from IPython.display import display

## 1. Load the Datasets

Two datasets loaded from HuggingFace:
- **Fashion Style Instruct**: 3,190 style recommendations (body type + occasion -> outfit)
- **Fashion Product Images**: 44,000+ real products with names, categories, colors, etc.

In [ ]:
# Layer 1: Style recommendations (body type + occasion -> outfit advice)
style_dataset = load_dataset("neuralwork/fashion-style-instruct", split="train")
style_df = style_dataset.to_pandas()
print(f"Style dataset shape: {style_df.shape}")
print(f"Columns: {style_df.columns.tolist()}")
style_df.head(2)

In [ ]:
# Layer 2: Real fashion products with names, categories, colors, etc.
product_dataset = load_dataset("ashraq/fashion-product-images-small", split="train")
product_df = product_dataset.to_pandas()
print(f"Product dataset shape: {product_df.shape}")
print(f"Columns: {product_df.columns.tolist()}")
product_df.head(5)

## 2. Explore the Data

In [ ]:
print("STYLE DATASET")
print(f"Total entries: {len(style_df)}")
print(f"\nExample INPUT (body + style):")
print(style_df['input'].iloc[0][:200])
print(f"\nExample CONTEXT (occasion):")
print(style_df['context'].iloc[0])
print(f"\nExample COMPLETION (outfit recommendation):")
print(style_df['completion'].iloc[0][:500])

In [ ]:
print("PRODUCT DATASET")
print(f"Total products: {len(product_df)}")

# Remove rows with missing key fields
product_df_clean = product_df.dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)
print(f"Products after cleaning: {len(product_df_clean)}")

print(f"\nGender distribution:")
print(product_df_clean['gender'].value_counts())
print(f"\nTop 10 article types:")
print(product_df_clean['articleType'].value_counts().head(10))
print(f"\nUsage categories:")
print(product_df_clean['usage'].value_counts())
print(f"\nSeasons:")
print(product_df_clean['season'].value_counts())

print(f"\nExample product names:")
for i in range(5):
    print(f"  {product_df_clean['productDisplayName'].iloc[i]}")

In [ ]:
# Preview some product images from the dataset
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    row = product_df_clean.iloc[i]
    img = product_dataset[int(product_df_clean.iloc[i].name)]['image']
    ax.imshow(img)
    ax.set_title(row['productDisplayName'][:30], fontsize=8)
    ax.axis('off')
plt.suptitle("Sample Products from the Catalog", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build the Index

*For products, we combine all text metadata fields (gender, category, article type, color, season, usage, and product name) into a single searchable text field. This is the text-based indexing strategy, images are available in the dataset but indexing and retrieval are performed on text only.*

In [ ]:
Doc = namedtuple('Doc', ['doc_id', 'text'])

# --- Style docs: combine input + context + completion into one document ---
style_docs = []
for idx, row in style_df.iterrows():
    text = f"Body: {row['input']}. Occasion: {row['context']}. Recommendation: {row['completion']}"
    style_docs.append(Doc(doc_id=str(idx), text=text))

print(f"# of style documents: {len(style_docs)}")

# --- Product docs: combine metadata fields into a rich text document ---
product_docs = []
for idx, row in product_df_clean.iterrows():
    parts = []
    for field in ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'usage']:
        val = row.get(field)
        if pd.notna(val):
            parts.append(str(val))
    name = str(row['productDisplayName'])
    text = " ".join(parts) + " - " + name
    product_docs.append(Doc(doc_id=str(idx), text=text))

print(f"# of product documents: {len(product_docs)}")

In [ ]:
print("STYLE DOC EXAMPLE")
print(style_docs[0].doc_id, style_docs[0].text[:300])
print()
print("PRODUCT DOC EXAMPLES")
for doc in product_docs[:5]:
    print(doc.doc_id, doc.text)

## Tokenizer and Inverted Index

In [ ]:
def simple_tokenize(text: str):
    return text.lower().split()

assert simple_tokenize("Hello World") == ["hello", "world"]

In [ ]:
#  Style Index
style_inverted_index = defaultdict(list)
style_corpus = {}

for doc in style_docs:
    style_corpus[doc.doc_id] = doc.text
    tokens = simple_tokenize(doc.text)
    unique_tokens = set(tokens)
    for token in unique_tokens:
        style_inverted_index[token].append(doc.doc_id)

print(f"Style index: {len(style_inverted_index)} unique terms.")

# Product Index
product_inverted_index = defaultdict(list)
product_corpus = {}

for doc in product_docs:
    product_corpus[doc.doc_id] = doc.text
    tokens = simple_tokenize(doc.text)
    unique_tokens = set(tokens)
    for token in unique_tokens:
        product_inverted_index[token].append(doc.doc_id)

print(f"Product index: {len(product_inverted_index)} unique terms.")

## Boolean Search

In [ ]:
def boolean_search(query, index):
    query_tokens = simple_tokenize(query)
    if not query_tokens:
        return []

    results = set(index.get(query_tokens[0], []))
    for token in query_tokens[1:]:
        results = results.intersection(set(index.get(token, [])))

    return list(results)

# Test on products
sample_query = "black formal shoes"
hits = boolean_search(sample_query, product_inverted_index)
print(f"Boolean hits for '{sample_query}': {len(hits)}")
for pid in hits[:5]:
    print(f"  {product_corpus[pid]}")

## BM25 Retrieval

In [ ]:
# Style BM25
style_tokenized = [simple_tokenize(doc.text) for doc in style_docs]
style_bm25 = BM25Okapi(style_tokenized)

def bm25_search_styles(query, n=5):
    tokenized_query = simple_tokenize(query)
    scores = style_bm25.get_scores(tokenized_query)
    top_n_indices = scores.argsort()[-n:][::-1]
    return [style_docs[i] for i in top_n_indices]

#  Product BM25
product_tokenized = [simple_tokenize(doc.text) for doc in product_docs]
product_bm25 = BM25Okapi(product_tokenized)

def bm25_search_products(query, n=5):
    tokenized_query = simple_tokenize(query)
    scores = product_bm25.get_scores(tokenized_query)
    top_n_indices = scores.argsort()[-n:][::-1]
    return [product_docs[i] for i in top_n_indices]

# Test
print("BM25 Product Search: 'black formal dress women'")
results = bm25_search_products("black formal dress women")
for i, res in enumerate(results):
    print(f"Rank {i+1}: {res.text}")

## Semantic Embedding with Sentence Transformers

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
style_texts = [doc.text for doc in style_docs]
style_embeddings = model.encode(style_texts, show_progress_bar=True)
print(f"Style Embedding Shape: {style_embeddings.shape}")

product_texts = [doc.text for doc in product_docs]
product_embeddings = model.encode(product_texts, show_progress_bar=True, batch_size=64)
print(f"Product Embedding Shape: {product_embeddings.shape}")

In [ ]:
def semantic_search_styles(query, n=5):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, style_embeddings)[0]
    top_n = scores.argsort(descending=True)[:n]
    return [(style_docs[i], scores[i].item()) for i in top_n]

def semantic_search_products(query, n=5):
    query_embedding = model.encode(query)
    scores = util.cos_sim(query_embedding, product_embeddings)[0]
    top_n = scores.argsort(descending=True)[:n]
    return [(product_docs[i], scores[i].item()) for i in top_n]

# Test
print("Semantic Product Search: 'elegant black evening dress'")
results = semantic_search_products("elegant black evening dress")
for i, (doc, score) in enumerate(results):
    print(f"Rank {i+1} (score: {score:.4f}): {doc.text}")

In [ ]:
test_sentences = [
    "women red wrap dress formal event",
    "ladies crimson draped gown cocktail party",
    "men blue casual running shoes sports",
]

test_embeddings = model.encode(test_sentences)

sim_fashion = util.cos_sim(test_embeddings[0], test_embeddings[1])
sim_mixed = util.cos_sim(test_embeddings[0], test_embeddings[2])

print(f"Similarity (red wrap dress vs crimson gown): {sim_fashion.item():.4f}")
print(f"Similarity (red wrap dress vs running shoes): {sim_mixed.item():.4f}")

## UMAP Visualization

In [ ]:
sample_size = min(200, len(style_docs))
sample_indices = np.random.choice(len(style_docs), sample_size, replace=False)
sample_embeddings_umap = style_embeddings[sample_indices]

reducer = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(sample_embeddings_umap)

plt.figure(figsize=(10, 7))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], color='navy', s=30)
plt.title("UMAP Projection of Style Recommendation Embeddings (sample)")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
sample_size_prod = min(500, len(product_docs))
sample_prod_indices = np.random.choice(len(product_docs), sample_size_prod, replace=False)
sample_prod_embeddings = product_embeddings[sample_prod_indices]

reducer2 = umap.UMAP(n_neighbors=10, min_dist=0.1, n_components=2, random_state=42)
prod_2d = reducer2.fit_transform(sample_prod_embeddings)

categories = [product_df_clean.iloc[i]['masterCategory'] for i in sample_prod_indices]
cat_colors = {
    'Apparel': 'red', 'Accessories': 'blue', 'Footwear': 'green',
    'Personal Care': 'purple', 'Free Items': 'orange',
    'Sporting Goods': 'brown', 'Home': 'pink'
}

plt.figure(figsize=(12, 8))
for cat in cat_colors:
    mask = [c == cat for c in categories]
    if any(mask):
        pts = prod_2d[mask]
        plt.scatter(pts[:, 0], pts[:, 1], c=cat_colors[cat], label=cat, s=30, alpha=0.6)

plt.legend()
plt.title("UMAP Projection of Product Embeddings (colored by Master Category)")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import plotly.express as px

sample_labels = []
for i in sample_indices:
    inp = style_df.iloc[i]['input'][:60]
    ctx = style_df.iloc[i]['context'][:40]
    sample_labels.append(f"{inp}... | {ctx}")

fig = px.scatter(
    x=embedding_2d[:, 0],
    y=embedding_2d[:, 1],
    hover_name=sample_labels,
    title="UMAP - Style Recommendations (hover to see details)"
)
fig.show()

## The Complete System: StyleFinder

A two-layer IR system:
- **Layer 1** uses semantic search on the style dataset to find outfit recommendations matching the user's body type, style, and occasion.
- **Layer 2** uses semantic search on the product catalog to find real products that match the recommended outfit pieces.

Product images from the dataset are displayed alongside text results as a visual aid, but all indexing and retrieval is performed on text fields only.

In [ ]:
def stylefinder(query, n_styles=3, n_products=5):
    """
    Two-layer IR system:
    Layer 1: Finds outfit recommendations based on body profile + occasion
    Layer 2: Finds real products matching the recommended outfit pieces
    """

    print(f"STYLEFINDER QUERY: {query}")

    # LAYER 1: Style Recommendations
    print(f"\nLAYER 1: Style Recommendations")

    style_results = semantic_search_styles(query, n=n_styles)

    for i, (doc, score) in enumerate(style_results):
        row = style_df.iloc[int(doc.doc_id)]
        print(f"\n  Recommendation {i+1} (score: {score:.4f}):")
        print(f"  Profile: {row['input'][:120]}...")
        print(f"  Occasion: {row['context']}")
        completion = row['completion']
        first_end = completion.find("Outfit Combination 2")
        if first_end == -1:
            first_end = completion.find("Outfit 2")
        if first_end == -1:
            first_end = completion.find("2. Outfit")
        if first_end == -1:
            first_end = 500
        print(f"  Top Outfit: {completion[:first_end].strip()}")

    # LAYER 2: Product Search with Images
    print(f"\n\n LAYER 2: Matching Products")

    best_style = style_df.iloc[int(style_results[0][0].doc_id)]
    recommendation_text = best_style['completion'][:300]
    product_query = query + " " + recommendation_text

    product_results = semantic_search_products(product_query, n=n_products)

    # Display products with images
    fig, axes = plt.subplots(1, n_products, figsize=(3 * n_products, 3.5))
    if n_products == 1:
        axes = [axes]

    for i, (doc, score) in enumerate(product_results):
        row = product_df_clean.iloc[int(doc.doc_id)]
        print(f"\n  Product {i+1} (score: {score:.4f}):")
        print(f"  Name: {row['productDisplayName']}")
        print(f"  Type: {row['articleType']} | Color: {row['baseColour']} | Season: {row['season']} | Usage: {row['usage']}")

        # Show product image from original dataset
        img = product_dataset[int(product_df_clean.iloc[int(doc.doc_id)].name)]['image']
        axes[i].imshow(img)
        axes[i].set_title(f"{row['productDisplayName'][:25]}...\nscore: {score:.3f}", fontsize=8)
        axes[i].axis('off')

    plt.suptitle(f"Products for: {query[:60]}...", fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return style_results, product_results

## Testing the StyleFinder

In [ ]:
# Query 1: Wedding guest
stylefinder("I have an hourglass figure and I am looking for something elegant to wear to a summer wedding")

In [ ]:
# Query 2: Job interview
stylefinder("I am a plus-size woman preparing for a job interview and I want to look polished and confident")

In [ ]:
# Query 3: Casual brunch
stylefinder("I am a tall man with broad shoulders looking for a casual outfit for a weekend brunch")

In [ ]:
# Query 4: Art exhibition
stylefinder("I am a skinny non-binary person who prefers minimalist gender-neutral clothing for an art exhibition")

In [ ]:
# Query 5: Hiking
stylefinder("I need a comfortable and practical outfit for hiking in cool weather");

## Comparison: BM25 vs Semantic Search on 10 Evaluation Queries

We test 7 realistic queries and 3 adversarial queries designed to expose system limitations.

In [ ]:
queries = [
    # Realistic queries
    "flattering wedding guest dress that cinches at the waist",
    "plus-size woman looking for professional interview outfit",
    "tall athletic man going to a casual date night",
    "non-binary person minimalist outfit for a gallery opening",
    "pear-shaped woman looking for a brunch outfit",
    "comfortable hiking outfit for cool autumn weather",
    "hourglass figure looking for an elegant evening gown",
    # Adversarial queries (test system limits)
    "outfit for wheelchair user at a formal event",
    "budget friendly outfit under 50 dollars",
    "sustainable eco-friendly clothing for a date night",
]

print("COMPARISON: BM25 vs SEMANTIC SEARCH")

for q in queries:
    print(f"QUERY: {q}")

    # Boolean
    bool_hits_style = boolean_search(q, style_inverted_index)
    bool_hits_product = boolean_search(q, product_inverted_index)
    print(f"\nBoolean hits -> Styles: {len(bool_hits_style)} | Products: {len(bool_hits_product)}")

    # BM25 Style Top 1
    print(f"\n BM25 Style Top 1")
    bm25_res = bm25_search_styles(q, n=1)
    for res in bm25_res:
        print(f"  {res.text[:150]}...")

    # Semantic Style Top 1
    print(f"\n Semantic Style Top 1")
    sem_res = semantic_search_styles(q, n=1)
    for doc, score in sem_res:
        print(f"  (score: {score:.4f}): {doc.text[:150]}...")

    # BM25 Product Top 3
    print(f"\n BM25 Product Top 3")
    bm25_prod = bm25_search_products(q, n=3)
    for i, res in enumerate(bm25_prod):
        print(f"  {i+1}. {res.text}")

    # Semantic Product Top 3
    print(f"\n Semantic Product Top 3")
    sem_prod = semantic_search_products(q, n=3)
    for i, (doc, score) in enumerate(sem_prod):
        print(f"  {i+1}. (score: {score:.4f}): {doc.text}")

## Other Query Testings

These queries test concepts absent from the datasets: wheelchair accessibility, budget constraints, and sustainability. The system should ideally indicate low confidence, but instead returns results with misleadingly high scores.

In [ ]:
adversarial_queries = [
    "outfit for wheelchair user at a formal event",
    "budget friendly outfit under 50 dollars",
    "sustainable eco-friendly clothing for a date night",
]

for q in adversarial_queries:
    stylefinder(q, n_styles=2, n_products=3)